# Quantitative Momentum Strategy

"Momentum investing" means investing in the stocks that have increased in price the most.

For this project, we're going to build an investing strategy that selects the 50 stocks with the highest price momentum. From there, we will calculate recommended trades for an equal-weight portfolio of these 50 stocks.


## Library Imports

The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [1]:
import numpy as np
import pandas as pd
import requests 
import xlsxwriter 
import math 
from io import StringIO
from scipy import stats 
import yfinance as yf

## Importing Our List of Stocks

As before, we'll need to import our list of stocks and our API token before proceeding. Make sure the `.csv` file is still in your working directory and import it with the following command:

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)

# Wrap response.text in StringIO()
# This prevents the FileNotFoundError by telling pandas: "This is a stream of text"
data_stream = StringIO(response.text)
sp500_table = pd.read_html(data_stream)

stocks = sp500_table[0]
stocks.rename(columns={'Symbol': 'Ticker'}, inplace=True)
stocks['Ticker'] = stocks['Ticker'].str.replace('.', '-', regex=False)

## Making Our First API Call

It's now time to make the first version of our momentum screener!

We need to get one-year price returns for each stock in the universe. Here's how.

In [3]:
symbol = 'AAPL' 
api_url = yf.Ticker(symbol) 
data = api_url.info 
data 

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

## Parsing Our API Call

This API call has all the information we need. We can parse it using the same square-bracket notation as in the first project of this course. Here is an example.

In [4]:
data.get('fiftyTwoWeekChangePercent')

41.157818

## Executing A Batch API Call & Building Our DataFrame

Just like in our first project, it's now time to execute several batch API calls and add the information we need to our DataFrame.

We'll start by running the following code cell, which contains some code we already built last time that we can re-use for this project. More specifically, it contains a function called `chunks` that we can use to divide our list of securities into groups of 100.

In [5]:
# # Function sourced from 
# # https://stackoverflow.com/questions/312443/how-do-you-split-a-list-into-evenly-sized-chunks
# def chunks(lst, n):
#     """Yield successive n-sized chunks from lst."""
#     for i in range(0, len(lst), n):
#         yield lst[i:i + n]   
        
# symbol_groups = list(chunks(stocks['Ticker'], 100))
# symbol_strings = []
# for i in range(0, len(symbol_groups)):
#     symbol_strings.append(','.join(symbol_groups[i]))
# #     print(symbol_strings[i])

# my_columns = ['Ticker', 'Price', 'One-Year Price Return', 'Number of Shares to Buy']

Now we need to create a blank DataFrame and add our data to the data frame one-by-one.

In [6]:
# price = data.get('currentPrice')
# market_cap = data.get('marketCap')

# my_columns = [ 'Ticker', 'Stock Price', 'Market Capitalisation', 'Number of Shares to Buy'] 
# final_dataframe = pd.DataFrame(columns = my_columns)  

# new_row = pd.Series([symbol, price, market_cap, 'N/A'], index = my_columns).to_frame().T
# final_dataframe = pd.concat([final_dataframe, new_row], ignore_index = True) 

# final_dataframe = pd.DataFrame(columns = my_columns) 
# for stock in stocks['Ticker']: 
#     api_url = yf.Ticker(stock) 
#     data = api_url.info 
#     new_row = pd.Series([stock, data.get('currentPrice'), data.get('marketCap'), 'N/A'], index = my_columns).to_frame().T 
#     final_dataframe = pd.concat([final_dataframe, new_row], ignore_index = True) 


# 1. Initialize the Tickers object with all symbols at once
# stocks['Ticker'] is your list from Wikipedia
tickers_list = stocks['Ticker'].tolist()
tickers_data = yf.Tickers(tickers_list)

rows_list = []

# print("Fetching data... this may take a minute.")

# 2. Loop through the tickers to get 'info'
# We use rows_list to avoid the slow pd.concat inside a loop
for symbol in tickers_list:
    try:
        # Accessing the individual ticker object from our batch
        ticker_info = tickers_data.tickers[symbol].info
        
        price = ticker_info.get('currentPrice') or ticker_info.get('regularMarketPrice')
        # market_cap = ticker_info.get('marketCap')
        one_year_price_return = ticker_info.get('fiftyTwoWeekChangePercent') 
        
        if price:
            rows_list.append({
                'Ticker': symbol,
                'Stock Price': price,
                'One-Year Price Return': one_year_price_return,
                'Number of Shares to Buy': 'N/A'
            })
    except Exception as e:
        # This skips tickers that might have been delisted mid-day or have errors
        continue

# 3. Create the final DataFrame in one go
final_dataframe = pd.DataFrame(rows_list)

final_dataframe 



,Ticker,Stock Price,One-Year Price Return,Number of Shares to Buy
0,MMM,145.12,-5.218470,N/A
1,AOS,57.97,-17.796368,N/A
2,ABT,84.90,-37.017803,N/A
3,ABBV,210.77,14.536465,N/A
4,ACN,163.99,-48.380493,N/A
...,...,...,...,...
498,XYL,109.44,-14.520037,N/A
499,YUM,150.63,1.728916,N/A
500,ZBRA,258.10,-14.144105,N/A
501,ZBH,82.65,-14.749872,N/A


## Removing Low-Momentum Stocks

The investment strategy that we're building seeks to identify the 50 highest-momentum stocks in the S&P 500.

Because of this, the next thing we need to do is remove all the stocks in our DataFrame that fall below this momentum threshold. We'll sort the DataFrame by the stocks' one-year price return, and drop all stocks outside the top 50.


In [7]:
final_dataframe.sort_values('One-Year Price Return', ascending = False, inplace = True) 
final_dataframe = final_dataframe[:51] 
final_dataframe.reset_index(drop = True, inplace = True) 
final_dataframe 

,Ticker,Stock Price,One-Year Price Return,Number of Shares to Buy
0,SNDK,1382.72,3319.189000,N/A
1,LITE,1001.81,1185.195700,N/A
2,WDC,489.15,878.495670,N/A
3,MU,776.01,691.846900,N/A
4,STX,804.76,646.599850,N/A
5,CIEN,591.57,628.175800,N/A
6,SATS,135.11,477.640000,N/A
7,INTC,115.93,435.226200,N/A
8,COHR,404.94,415.190830,N/A
9,FIX,2042.36,332.730930,N/A


## Calculating the Number of Shares to Buy

Just like in the last project, we now need to calculate the number of shares we need to buy. The one change we're going to make is wrapping this functionality inside a function, since we'll be using it again later in this Jupyter Notebook.

Since we've already done most of the work on this, try to complete the following two code cells without watching me do it first!

In [8]:
# def portfolio_input(): 
#     global portfoli_size 
#     portfolio_size = input("Enter the value of your portfolio:") 

#     try: 
#         val = float(portfolio_size) 
#     except ValueError: 
#         print("That's not a number! \n Try again:") 
#         portfolio_size = input("Ener the value of portfoli:") 


# portfolio_input()
# print(portfolio_size) 

def portfolio_input():
    while True:
        portfolio_size = input("Enter the value of your portfolio: ")

        try:
            return float(portfolio_size)
        except ValueError:
            print("That's not a number! Try again.")

portfolio_size = portfolio_input()
print(portfolio_size)

Enter the value of your portfolio:  10000000


10000000.0


In [9]:
# Force the column to be numeric so it can hold the result of math.floor 
final_dataframe['Number of Shares to Buy'] = pd.to_numeric(final_dataframe['Number of Shares to Buy'], errors='coerce')

position_size = float(portfolio_size) / len(final_dataframe.index) 
for i in range(0, len(final_dataframe.index)): 
    price = final_dataframe.loc[i, 'Stock Price'] 
    final_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / price)

final_dataframe 

,Ticker,Stock Price,One-Year Price Return,Number of Shares to Buy
0,SNDK,1382.72,3319.189000,141.0
1,LITE,1001.81,1185.195700,195.0
2,WDC,489.15,878.495670,400.0
3,MU,776.01,691.846900,252.0
4,STX,804.76,646.599850,243.0
5,CIEN,591.57,628.175800,331.0
6,SATS,135.11,477.640000,1451.0
7,INTC,115.93,435.226200,1691.0
8,COHR,404.94,415.190830,484.0
9,FIX,2042.36,332.730930,96.0


## Building a Better (and More Realistic) Momentum Strategy

Real-world quantitative investment firms differentiate between "high quality" and "low quality" momentum stocks:

* High-quality momentum stocks show "slow and steady" outperformance over long periods of time
* Low-quality momentum stocks might not show any momentum for a long time, and then surge upwards.

The reason why high-quality momentum stocks are preferred is because low-quality momentum can often be cause by short-term news that is unlikely to be repeated in the future (such as an FDA approval for a biotechnology company).

To identify high-quality momentum, we're going to build a strategy that selects stocks from the highest percentiles of: 

* 1-month price returns
* 3-month price returns
* 6-month price returns
* 1-year price returns

Let's start by building our DataFrame. You'll notice that I use the abbreviation `hqm` often. It stands for `high-quality momentum`.

In [10]:
# hqm_columns = [ 
#     'Ticker', 
#     'Price', 
#     'Number of Shares to Buy', 
#     'One-Year Price Return', 
#     'One-Year Return Percentile', 
#     'Six-Month Price Return', 
#     'Six-Month Return Percentile', 
#     'Three-Month Price Return', 
#     'Three-Month Return Percentile', 
#     'One-Month Price Return', 
#     'One-Month Return Percentile', 
#     'HQM Score'
# ] 

# hqm_dataframe = pd.DataFrame(columns = hqm_columns) 

# for symbol in tickers_list:
#     try:
#         # Accessing the individual ticker object from our batch
#         ticker_info = tickers_data.tickers[symbol].info
#         price = ticker_info.get('currentPrice') or ticker_info.get('regularMarketPrice')
#         one_year_price_return = ticker_info.get('fiftyTwoWeekChangePercent') 
#         month_6_Change_Percent = ticker_infor.get('twoHundredDayAverageChangePercent') 
#         month_3_Change_Percent = ticker_info.get('fiftyDayAverageChangePercent') 
#         month_1_Change_Percent = ticket_info.get('') 
        
#         if price:
#             rows_list.append({
#                 'Ticker': symbol,
#                 'Stock Price': price,
#                 'One-Year Price Return': one_year_price_return,
#                 'Number of Shares to Buy': 'N/A'
#             })
#     except Exception as e:
#         # This skips tickers that might have been delisted mid-day or have errors
#         continue

import datetime

# 1. Define your universe and timeframes
tickers = stocks['Ticker'].tolist()
end_date = datetime.datetime.now()
start_date = end_date - datetime.timedelta(days=380) # Buffer to ensure we have a full year

# 2. Download historical data for the entire S&P 500 at once
# This is the "True" batch method for performance
print("Downloading historical data...")
ohlc_data = yf.download(tickers, start=start_date, end=end_date, interval='1d')['Close']

hqm_list = []

print("Calculating momentum metrics...")
for symbol in tickers:
    try:
        # Get price series for this stock and drop missing values
        series = ohlc_data[symbol].dropna()
        if len(series) < 250: continue # Skip if not enough history
        
        current_price = series.iloc[-1]
        
        # Calculate returns (Current Price / Historical Price) - 1
        # We use approximate trading days: 1yr=252, 6mo=126, 3mo=63, 1mo=21
        return_1yr = (current_price / series.iloc[-252]) - 1 if len(series) >= 252 else 0
        return_6mo = (current_price / series.iloc[-126]) - 1 if len(series) >= 126 else 0
        return_3mo = (current_price / series.iloc[-63]) - 1 if len(series) >= 63 else 0
        return_1mo = (current_price / series.iloc[-21]) - 1 if len(series) >= 21 else 0
        
        hqm_list.append({
            'Ticker': symbol,
            'Price': current_price,
            'Number of Shares to Buy': 'N/A',
            'One-Year Price Return': return_1yr,
            'One-Year Return Percentile': 'N/A',
            'Six-Month Price Return': return_6mo,
            'Six-Month Return Percentile': 'N/A',
            'Three-Month Price Return': return_3mo,
            'Three-Month Return Percentile': 'N/A',
            'One-Month Price Return': return_1mo,
            'One-Month Return Percentile': 'N/A',
            'HQM Score': 'N/A'
        })
    except Exception:
        continue

hqm_dataframe = pd.DataFrame(hqm_list)
hqm_dataframe

[*********************100%***********************]  503 of 503 completed


Calculating momentum metrics...


,Ticker,Price,Number of Shares to Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,MMM,145.119995,N/A,-0.000580,N/A,-0.144274,N/A,-0.155395,N/A,-0.036068,N/A,N/A
1,AOS,57.970001,N/A,-0.134018,N/A,-0.117254,N/A,-0.275421,N/A,-0.078704,N/A,N/A
2,ABT,84.900002,N/A,-0.326117,N/A,-0.333447,N/A,-0.241812,N/A,-0.110715,N/A,N/A
3,ABBV,210.770004,N/A,0.226536,N/A,-0.081471,N/A,-0.081993,N/A,0.008517,N/A,N/A
4,ACN,163.990005,N/A,-0.481265,N/A,-0.329166,N/A,-0.262451,N/A,-0.154691,N/A,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...
497,XYL,109.440002,N/A,-0.116941,N/A,-0.265219,N/A,-0.142704,N/A,-0.125948,N/A,N/A
498,YUM,150.630005,N/A,0.057649,N/A,0.017881,N/A,-0.057581,N/A,-0.055730,N/A,N/A
499,ZBRA,258.100006,N/A,-0.137568,N/A,0.009425,N/A,-0.024639,N/A,0.107535,N/A,N/A
500,ZBH,82.650002,N/A,-0.123947,N/A,-0.077033,N/A,-0.140752,N/A,-0.127981,N/A,N/A


## Calculating Momentum Percentiles

We now need to calculate momentum percentile scores for every stock in the universe. More specifically, we need to calculate percentile scores for the following metrics for every stock:

* `One-Year Price Return`
* `Six-Month Price Return`
* `Three-Month Price Return`
* `One-Month Price Return`

Here's how we'll do this:

In [11]:
# Force the column to be numeric so it can hold the result of math.floor 
hqm_dataframe['One-Year Return Percentile'] = pd.to_numeric(hqm_dataframe['One-Year Return Percentile'], errors='coerce')
hqm_dataframe['Six-Month Return Percentile'] = pd.to_numeric(hqm_dataframe['Six-Month Return Percentile'], errors='coerce')
hqm_dataframe['Three-Month Return Percentile'] = pd.to_numeric(hqm_dataframe['Three-Month Return Percentile'], errors='coerce')
hqm_dataframe['One-Month Return Percentile'] = pd.to_numeric(hqm_dataframe['One-Month Return Percentile'], errors='coerce')

time_periods = [
                'One-Year',
                'Six-Month',
                'Three-Month',
                'One-Month'
                ]

for row in hqm_dataframe.index:
    for time_period in time_periods:
        hqm_dataframe.loc[row, f'{time_period} Return Percentile'] = stats.percentileofscore(hqm_dataframe[f'{time_period} Price Return'], hqm_dataframe.loc[row, f'{time_period} Price Return'])/100

# Print each percentile score to make sure it was calculated properly
for time_period in time_periods:
    print(hqm_dataframe[f'{time_period} Return Percentile'])

#Print the entire DataFrame    
hqm_dataframe

0      0.366534
1      0.215139
2      0.095618
3      0.643426
4      0.029880
         ...   
497    0.229084
498    0.438247
499    0.207171
500    0.221116
501    0.025896
Name: One-Year Return Percentile, Length: 502, dtype: float64
0      0.219124
1      0.243028
2      0.045817
3      0.286853
4      0.049801
         ...   
497    0.099602
498    0.464143
499    0.444223
500    0.292829
501    0.029880
Name: Six-Month Return Percentile, Length: 502, dtype: float64
0      0.181275
1      0.049801
2      0.075697
3      0.336653
4      0.063745
         ...   
497    0.199203
498    0.398406
499    0.503984
500    0.209163
501    0.005976
Name: Three-Month Return Percentile, Length: 502, dtype: float64
0      0.390438
1      0.221116
2      0.147410
3      0.607570
4      0.051793
         ...   
497    0.109562
498    0.290837
499    0.836653
500    0.103586
501    0.003984
Name: One-Month Return Percentile, Length: 502, dtype: float64


,Ticker,Price,Number of Shares to Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,MMM,145.119995,N/A,-0.000580,0.366534,-0.144274,0.219124,-0.155395,0.181275,-0.036068,0.390438,N/A
1,AOS,57.970001,N/A,-0.134018,0.215139,-0.117254,0.243028,-0.275421,0.049801,-0.078704,0.221116,N/A
2,ABT,84.900002,N/A,-0.326117,0.095618,-0.333447,0.045817,-0.241812,0.075697,-0.110715,0.147410,N/A
3,ABBV,210.770004,N/A,0.226536,0.643426,-0.081471,0.286853,-0.081993,0.336653,0.008517,0.607570,N/A
4,ACN,163.990005,N/A,-0.481265,0.029880,-0.329166,0.049801,-0.262451,0.063745,-0.154691,0.051793,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...
497,XYL,109.440002,N/A,-0.116941,0.229084,-0.265219,0.099602,-0.142704,0.199203,-0.125948,0.109562,N/A
498,YUM,150.630005,N/A,0.057649,0.438247,0.017881,0.464143,-0.057581,0.398406,-0.055730,0.290837,N/A
499,ZBRA,258.100006,N/A,-0.137568,0.207171,0.009425,0.444223,-0.024639,0.503984,0.107535,0.836653,N/A
500,ZBH,82.650002,N/A,-0.123947,0.221116,-0.077033,0.292829,-0.140752,0.209163,-0.127981,0.103586,N/A


## Calculating the HQM Score

We'll now calculate our `HQM Score`, which is the high-quality momentum score that we'll use to filter for stocks in this investing strategy.

The `HQM Score` will be the arithmetic mean of the 4 momentum percentile scores that we calculated in the last section.

To calculate arithmetic mean, we will use the `mean` function from Python's built-in `statistics` module.

In [12]:
# Force the column to be numeric so it can hold the result of float 
hqm_dataframe['HQM Score'] = pd.to_numeric(hqm_dataframe['HQM Score'], errors='coerce')

from statistics import mean 

for row in hqm_dataframe.index: 
    momentum_percentiles = [] 
    for time_period in time_periods: 
        momentum_percentiles.append(hqm_dataframe.loc[row, f'{time_period} Return Percentile'])
    hqm_dataframe.loc[row, 'HQM Score'] = mean(momentum_percentiles)

## Selecting the 50 Best Momentum Stocks

As before, we can identify the 50 best momentum stocks in our universe by sorting the DataFrame on the `HQM Score` column and dropping all but the top 50 entries.

In [13]:
hqm_dataframe.sort_values(by = 'HQM Score', ascending = False) 
hqm_dataframe = hqm_dataframe[:51] 
hqm_dataframe 

,Ticker,Price,Number of Shares to Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,MMM,145.119995,N/A,-0.000580,0.366534,-0.144274,0.219124,-0.155395,0.181275,-0.036068,0.390438,0.289343
1,AOS,57.970001,N/A,-0.134018,0.215139,-0.117254,0.243028,-0.275421,0.049801,-0.078704,0.221116,0.182271
2,ABT,84.900002,N/A,-0.326117,0.095618,-0.333447,0.045817,-0.241812,0.075697,-0.110715,0.147410,0.091135
3,ABBV,210.770004,N/A,0.226536,0.643426,-0.081471,0.286853,-0.081993,0.336653,0.008517,0.607570,0.468625
4,ACN,163.990005,N/A,-0.481265,0.029880,-0.329166,0.049801,-0.262451,0.063745,-0.154691,0.051793,0.048805
5,ADBE,237.009995,N/A,-0.406689,0.059761,-0.296811,0.079681,-0.102133,0.278884,-0.044931,0.330677,0.187251
6,AMD,449.700012,N/A,2.820082,0.976096,0.737031,0.948207,1.169111,0.996016,0.616114,0.994024,0.978586
7,AES,14.460000,N/A,0.214329,0.621514,0.052709,0.531873,-0.100842,0.286853,0.009537,0.613546,0.513446
8,AFL,116.389999,N/A,0.144500,0.525896,0.026252,0.488048,0.018261,0.619522,0.023569,0.655378,0.572211
9,A,113.260002,N/A,0.023939,0.386454,-0.249463,0.115538,-0.097699,0.300797,-0.042199,0.344622,0.286853


## Calculating the Number of Shares to Buy

We'll use the `portfolio_input` function that we created earlier to accept our portfolio size. Then we will use similar logic in a `for` loop to calculate the number of shares to buy for each stock in our investment universe.

In [15]:
portfolio_input()

Enter the value of your portfolio:  10000000


10000000.0

In [16]:
# Force the column to be numeric so it can hold the result of float 
hqm_dataframe['Number of Shares to Buy'] = pd.to_numeric(hqm_dataframe['Number of Shares to Buy'], errors='coerce')

position_size = float(portfolio_size) / len(hqm_dataframe.index)
for i in range(0, len(hqm_dataframe['Ticker'])-1):
    hqm_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / hqm_dataframe['Price'][i])
hqm_dataframe

,Ticker,Price,Number of Shares to Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,MMM,145.119995,1351.0,-0.000580,0.366534,-0.144274,0.219124,-0.155395,0.181275,-0.036068,0.390438,0.289343
1,AOS,57.970001,3382.0,-0.134018,0.215139,-0.117254,0.243028,-0.275421,0.049801,-0.078704,0.221116,0.182271
2,ABT,84.900002,2309.0,-0.326117,0.095618,-0.333447,0.045817,-0.241812,0.075697,-0.110715,0.147410,0.091135
3,ABBV,210.770004,930.0,0.226536,0.643426,-0.081471,0.286853,-0.081993,0.336653,0.008517,0.607570,0.468625
4,ACN,163.990005,1195.0,-0.481265,0.029880,-0.329166,0.049801,-0.262451,0.063745,-0.154691,0.051793,0.048805
5,ADBE,237.009995,827.0,-0.406689,0.059761,-0.296811,0.079681,-0.102133,0.278884,-0.044931,0.330677,0.187251
6,AMD,449.700012,436.0,2.820082,0.976096,0.737031,0.948207,1.169111,0.996016,0.616114,0.994024,0.978586
7,AES,14.460000,13560.0,0.214329,0.621514,0.052709,0.531873,-0.100842,0.286853,0.009537,0.613546,0.513446
8,AFL,116.389999,1684.0,0.144500,0.525896,0.026252,0.488048,0.018261,0.619522,0.023569,0.655378,0.572211
9,A,113.260002,1731.0,0.023939,0.386454,-0.249463,0.115538,-0.097699,0.300797,-0.042199,0.344622,0.286853


## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

In [27]:
writer = pd.ExcelWriter('momentum_strategy.xlsx', engine='xlsxwriter')
hqm_dataframe.to_excel(writer, sheet_name='Momentum Strategy', index = False)

## Creating the Formats We'll Need For Our .xlsx File

You'll recall from our first project that formats include colors, fonts, and also symbols like % and $. We'll need four main formats for our Excel document:

* String format for tickers
* \$XX.XX format for stock prices
* \$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase

Since we already built our formats in the last section of this course, I've included them below for you. Run this code cell before proceeding.

In [28]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_template = writer.book.add_format(
        {
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

dollar_template = writer.book.add_format(
        {
            'num_format':'$0.00',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

integer_template = writer.book.add_format(
        {
            'num_format':'0.000000',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

percent_template = writer.book.add_format(
        {
            'num_format':'0.0%',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

In [29]:
column_formats = { 
                    'A': ['Ticker', string_template],
                    'B': ['Price', dollar_template],
                    'C': ['Number of Shares to Buy', integer_template],
                    'D': ['One-Year Price Return', percent_template],
                    'E': ['One-Year Return Percentile', percent_template],
                    'F': ['Six-Month Price Return', percent_template],
                    'G': ['Six-Month Return Percentile', percent_template],
                    'H': ['Three-Month Price Return', percent_template],
                    'I': ['Three-Month Return Percentile', percent_template],
                    'J': ['One-Month Price Return', percent_template],
                    'K': ['One-Month Return Percentile', percent_template],
                    'L': ['HQM Score', integer_template]
                    }

for column in column_formats.keys():
    writer.sheets['Momentum Strategy'].set_column(f'{column}:{column}', 20, column_formats[column][1])
    writer.sheets['Momentum Strategy'].write(f'{column}1', column_formats[column][0], string_template)

## Saving Our Excel Output

As before, saving our Excel output is very easy:

In [31]:
writer.close()